# HUMAN ACTIVITY RECOGNITION USING SMARTPHONE SENSOR DATA
## Project by SHAURYA THAKUR

**Framework:** TensorFlow / Keras  
**Model:** Long Short-Term Memory (LSTM)  
**Dataset:** UCI Human Activity Recognition Using Smartphones

### Objective
Classify six human activities from time-series measurements recorded by smartphone accelerometer and gyroscope sensors.

**Activities:** Walking, Walking Upstairs, Walking Downstairs, Sitting, Standing, Laying.

**Dataset credit:** Anguita et al., UCI Machine Learning Repository, *Human Activity Recognition Using Smartphones*. DOI: 10.24432/C54S4K.

This notebook is an independent implementation for practical demonstration. Preserve applicable dataset and source-code attribution.

## 1. Upload and extract the UCI HAR dataset

Upload the ZIP downloaded from the official UCI dataset page. The download may contain another ZIP named `UCI HAR Dataset.zip`; this cell extracts uploaded ZIP files recursively and locates the dataset folder.

In [ ]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()
extract_root = "/content/sha urya_deep_learning_har".replace(" ", "")
os.makedirs(extract_root, exist_ok=True)

# Extract each uploaded ZIP.
for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as zf:
            zf.extractall(extract_root)

# Extract any nested ZIP files, such as the inner UCI HAR Dataset.zip.
for root, dirs, filenames in os.walk(extract_root):
    for filename in filenames:
        if filename.lower().endswith(".zip"):
            nested_path = os.path.join(root, filename)
            nested_output = os.path.join(root, "nested_extracted")
            os.makedirs(nested_output, exist_ok=True)
            with zipfile.ZipFile(nested_path, "r") as zf:
                zf.extractall(nested_output)

# Find the directory containing raw inertial signal folders.
dataset_dir = None
for root, dirs, filenames in os.walk(extract_root):
    train_signal_dir = os.path.join(root, "train", "Inertial Signals")
    test_signal_dir = os.path.join(root, "test", "Inertial Signals")
    if os.path.isdir(train_signal_dir) and os.path.isdir(test_signal_dir):
        dataset_dir = root
        break

if dataset_dir is None:
    raise FileNotFoundError(
        "Could not locate train/test Inertial Signals folders. "
        "Upload the official UCI HAR ZIP."
    )

print("Dataset found:", dataset_dir)

## 2. Load raw time-series sensor signals

The UCI HAR dataset provides nine inertial signal channels for each sample: body acceleration (X, Y, Z), total acceleration (X, Y, Z), and body gyroscope (X, Y, Z).

Each sample contains 128 time steps. We combine the nine channels into an input tensor with shape **(number of samples, 128, 9)**.

In [ ]:
import numpy as np
import pandas as pd

signal_names = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_signals(split):
    signal_folder = os.path.join(dataset_dir, split, "Inertial Signals")
    channels = []
    for signal in signal_names:
        filename = os.path.join(signal_folder, f"{signal}_{split}.txt")
        values = pd.read_csv(filename, sep=r"\\s+", header=None).to_numpy()
        channels.append(values)
    # (9, samples, 128) -> (samples, 128, 9)
    return np.stack(channels, axis=-1).astype("float32")

X_train = load_signals("train")
X_test = load_signals("test")

y_train = pd.read_csv(
    os.path.join(dataset_dir, "train", "y_train.txt"),
    header=None
).iloc[:, 0].to_numpy() - 1

y_test = pd.read_csv(
    os.path.join(dataset_dir, "test", "y_test.txt"),
    header=None
).iloc[:, 0].to_numpy() - 1

print("Training sensor tensor:", X_train.shape)
print("Testing sensor tensor :", X_test.shape)
print("Training labels       :", y_train.shape)
print("Testing labels        :", y_test.shape)

assert X_train.shape == (7352, 128, 9)
assert X_test.shape == (2947, 128, 9)
print("Raw sensor data loaded and dimensions verified.")

## 3. Prepare labels and libraries

The six original activity labels are numbered 1–6. We converted them to 0–5 for the neural network's sparse categorical cross-entropy loss.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

activity_names = [
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
]

print("TensorFlow version:", tf.__version__)
print("Number of activity classes:", len(activity_names))
print("Input shape per sample:", X_train.shape[1:])

## 4. Build the LSTM deep-learning model

LSTM is a type of recurrent neural network (RNN) designed to learn patterns in sequential data. It is suitable for time-series sensor measurements because it can learn how motion changes across time steps.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(128, 9)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(6, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the LSTM model

The model learns from the training sensor sequences. Validation data is taken from the training set; the official test set is kept for final evaluation.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## 6. Evaluate on the official test dataset

The test set is separate from the training data. The accuracy printed below is the result of this actual Colab run.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

## 7. Plot training and validation performance

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("SHAURYA THAKUR - Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("LSTM Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 8. Classification report and confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

probabilities = model.predict(X_test, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

print(classification_report(
    y_test,
    y_pred,
    labels=list(range(6)),
    target_names=activity_names,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=list(range(6)))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=activity_names
).plot(ax=ax, xticks_rotation=40, cmap="Blues", colorbar=False)
plt.title("SHAURYA THAKUR - Activity Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Demonstrate one activity prediction

This predicts the activity for one sample from the official test set. It is an offline dataset example, not live sensing from a phone.

In [ ]:
sample_index = 0
sample = X_test[sample_index:sample_index + 1]

prediction_probabilities = model.predict(sample, verbose=0)[0]
predicted_class = int(np.argmax(prediction_probabilities))
actual_class = int(y_test[sample_index])

print("Project: Human Activity Recognition")
print("Prepared by: SHAURYA THAKUR")
print("Predicted activity:", activity_names[predicted_class])
print("Actual activity:   ", activity_names[actual_class])
print("Prediction confidence:", f"{prediction_probabilities[predicted_class] * 100:.2f}%")

## 10. Conclusion

This practical used raw smartphone inertial time-series signals to train an LSTM deep-learning model for six-class human activity recognition.

The notebook:
1. Loads the UCI HAR raw inertial signals.
2. Arranges sensor data into sequences of 128 time steps and 9 channels.
3. Builds and trains an LSTM neural network.
4. Evaluates the model on the official test split.
5. Displays accuracy curves, a classification report, a confusion matrix, and an example prediction.

### Limitations
- The demonstration uses a pre-collected dataset, not live readings from a smartphone.
- Test accuracy must be taken from the output generated by your own run.
- Training time depends on the Colab runtime.

### Dataset reference
UCI Machine Learning Repository — Human Activity Recognition Using Smartphones  
https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones  
DOI: https://doi.org/10.24432/C54S4K

**Project prepared by: SHAURYA THAKUR**